# Germplasm Exploration — brapi-py
Interactive notebook for exploring the BrAPI Germplasm endpoint.
Run each cell with **Shift+Enter** (or the ▶ button).

> **First time?** Select a Python kernel with the **Select Kernel** button (top-right).
> If `brapi-py` is not installed, run the pip cell below first.

In [ ]:
# Install / upgrade brapi-py if needed (remove the # to run)
# %pip install -e "c:/Projects/brapi-py"
import brapi
print("brapi-py version:", brapi.__version__)

## 1 — Connection
Edit the cell below with your server credentials, then run it.
The `with` block is optional here — the client stays open for the whole notebook.

In [ ]:
from brapi import BrapiClient

# ── Edit these ────────────────────────────────────────────────────────────
BASE_URL       = "https://brapi.example.com"
TOKEN_ENDPOINT = "https://auth.example.com/token"
CLIENT_ID      = "my-client"
CLIENT_SECRET  = "my-secret"
# ─────────────────────────────────────────────────────────────────────────

client = BrapiClient(
    base_url=BASE_URL,
    token_endpoint=TOKEN_ENDPOINT,
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET,
)
print("Client ready →", client)

## 2 — Search  (`POST /search/germplasm`)
Full-featured search. Handles both synchronous (200) and asynchronous (202 + polling) responses.

In [ ]:
# Basic search — all Wheat germplasm
df = (
    client.germplasm
    .by_crop(["Wheat"])
    .search()
    .to_df()
)
print(f"{len(df)} records returned")
df.head()

In [ ]:
# Chained filters — fork the same base query safely
base = client.germplasm.by_crop(["Wheat"])

triticum_df = base.genus(["Triticum"]).search().to_df()
hordeum_df  = base.genus(["Hordeum"]).search().to_df()   # base unchanged

print("Triticum:", len(triticum_df), "  Hordeum:", len(hordeum_df))

In [ ]:
# Bulk filter — apply many criteria at once
df = (
    client.germplasm
    .filter(
        common_crop_names=["Wheat"],
        genus=["Triticum"],
        species=["aestivum"],
        institute_codes=["DEU084"],
    )
    .search()
    .to_df()
)
df[["germplasmDbId", "germplasmName", "genus", "species", "instituteCode"]].head(10)

## 3 — List  (`GET /germplasm`)
Simpler endpoint — same filter state, mapped to query-string params.
Use when the server doesn't support the search endpoint, or for quick lookups.

In [ ]:
df = (
    client.germplasm
    .by_crop(["Wheat"])
    .genus(["Triticum"])
    .list()           # GET /germplasm?commonCropName=Wheat&genus=Triticum
    .to_df()
)
print(f"{len(df)} records")
df.head()

## 4 — Single-record CRUD
Get, create, update, and delete individual records.

In [ ]:
# GET /germplasm/{id}  — fetch one record by primary key
GERMPLASM_ID = "g-001"   # ← change to a real ID from the search above

g = client.germplasm.get_by_id(GERMPLASM_ID)
print(g.germplasmDbId, "|", g.germplasmName, "|", g.genus, g.species)

In [ ]:
from brapi.entities.germplasm import Germplasm

# POST /germplasm — create a single record
new_germplasm = Germplasm(
    germplasmDbId="",                          # server assigns
    germplasmName="NotebookTest-001",
    germplasmPUI="http://pui.example/nb-001",
    commonCropName="Wheat",
    genus="Triticum",
    species="aestivum",
)
created = client.germplasm.create(new_germplasm)
print("Created:", created.germplasmDbId, created.germplasmName)

In [ ]:
# PUT /germplasm/{id} — update an existing record
created.pedigree = "ParentA / ParentB"
updated = client.germplasm.update(created.germplasmDbId, created)
print("Updated pedigree:", updated.pedigree)

In [ ]:
# DELETE /germplasm/{id} — remove the record we just created
ok = client.germplasm.delete(created.germplasmDbId)
print("Deleted:", ok)

## 5 — Pipe transforms
`.pipe()` applies a function lazily — only one HTTP call is made regardless of how many pipe stages are chained.

In [ ]:
def with_cultivated_only(items):
    """Keep only cultivated material (biologicalStatusOfAccessionCode == '100')."""
    return [g for g in items if g.biologicalStatusOfAccessionCode == "100"]

def add_display_label(items):
    """Add a short label: 'Genus species (DbId)'."""
    for g in items:
        genus = g.genus or ""
        species = g.species or ""
        g.label = f"{genus} {species} ({g.germplasmDbId})".strip()
    return items

df = (
    client.germplasm
    .by_crop(["Wheat"])
    .search()
    .pipe(with_cultivated_only)
    .pipe(add_display_label)
    .to_df()
)
df[["germplasmDbId", "germplasmName", "label"]].head(10)

## 6 — DataFrame exploration
Useful pandas operations once you have a DataFrame back.

In [ ]:
df = client.germplasm.by_crop(["Wheat"]).search().to_df()

# What columns came back?
print("Columns:", df.columns.tolist())
print("\nShape:", df.shape)
df.head()

In [ ]:
import json, pandas as pd

# Breakdown by genus
df["genus"].value_counts().head(10)

In [ ]:
# Explode the synonyms JSON column into separate rows for analysis
if "synonyms" in df.columns:
    synonyms_df = (
        df[["germplasmDbId", "germplasmName", "synonyms"]]
        .dropna(subset=["synonyms"])
        .assign(synonyms=lambda d: d["synonyms"].apply(json.loads))
        .explode("synonyms")
        .assign(
            synonym_name=lambda d: d["synonyms"].apply(lambda x: x.get("synonym") if isinstance(x, dict) else None),
            synonym_type=lambda d: d["synonyms"].apply(lambda x: x.get("type") if isinstance(x, dict) else None),
        )
        .drop(columns="synonyms")
    )
    synonyms_df.head(10)
else:
    print("No synonyms column in this result set")

## 7 — Error handling

In [ ]:
import httpx

# 404 — record not found
try:
    client.germplasm.get_by_id("does-not-exist-xyz")
except httpx.HTTPStatusError as e:
    print(f"HTTP {e.response.status_code}: {e.request.url}")

# Async search timeout (demo — won't actually time out with a real server)
try:
    client.germplasm.by_crop(["Wheat"]).search(
        poll_interval=1.0,
        max_poll_attempts=1,   # force fast timeout for demo
    ).to_list()
except TimeoutError as e:
    print("TimeoutError:", e)

# ValueError from create_many with only 1 item
try:
    client.germplasm.create_many([new_germplasm])
except ValueError as e:
    print("ValueError:", e)

## 8 — Clean up
Close the HTTP client when done to release the connection pool.

In [ ]:
client.close()
print("Connection closed")